# 02 — Task 2.2: Unify All Transaction Sources

**Target Table:** `bfsi.silver_layer.unified_transactions`

**Objective:** Standardize Card, UPI, and NEFT/RTGS transactions into a single table with a common schema.

| Feature | Implementation |
|---------|----------------|
| Schema standardization | Common columns across all 3 channels |
| Currency conversion | Exchange-rate lookup for international card txns |
| Status normalization | SUCCESS, FAILED, PENDING, REVERSED |
| Cross-source dedup | Hash-based matching to flag duplicates |

**Common Columns:**  
`txn_id`, `customer_id`, `txn_channel`, `txn_amount_inr`, `txn_ts`, `status`, `masked_counterparty`, `merchant_id`, `is_international`, `country_code`

> Reads from **Silver masked tables** (Task 2.1 output) — Bronze remains untouched.

---
## Cell 1 — Configuration & Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, MapType
from datetime import datetime

# ── Catalog / Schema ───────────────────────────────────────────────────
CATALOG       = "bfsi"
SILVER_SCHEMA = "silver_layer"

# ── Source tables (Silver masked — from Task 2.1) ──────────────────────
CARD_TXN_TABLE  = f"{CATALOG}.{SILVER_SCHEMA}.card_transactions_masked"
UPI_TXN_TABLE   = f"{CATALOG}.{SILVER_SCHEMA}.upi_transactions_masked"
NEFT_RTGS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.neft_rtgs_transactions_clean"

# ── Target table ───────────────────────────────────────────────────────
UNIFIED_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.unified_transactions"

print(f"Task 2.2: Unify Transaction Sources")
print(f"Timestamp: {datetime.now().isoformat()}")
print(f"Target   : {UNIFIED_TABLE}")

---
## Cell 2 — Exchange Rate Lookup Table (Currency Conversion)

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  EXCHANGE RATE LOOKUP TABLE
#  Maps foreign currencies → INR for international card transactions
#  TODO: Replace with a live FX rate feed or Delta table in production
# ══════════════════════════════════════════════════════════════════════════

exchange_rates_data = [
    ("USD", 83.50),
    ("EUR", 91.20),
    ("GBP", 106.30),
    ("JPY", 0.56),
    ("AUD", 54.80),
    ("CAD", 62.10),
    ("SGD", 62.50),
    ("AED", 22.73),
    ("SAR", 22.27),
    ("CHF", 94.50),
    ("CNY", 11.50),
    ("HKD", 10.70),
    ("THB", 2.35),
    ("MYR", 17.80),
    ("INR", 1.00),   # Identity — domestic transactions
]

exchange_rate_schema = StructType([
    StructField("currency_code", StringType(), False),
    StructField("rate_to_inr", DoubleType(), False),
])

df_exchange_rates = spark.createDataFrame(exchange_rates_data, schema=exchange_rate_schema)

print(f"Exchange rate lookup table: {df_exchange_rates.count()} currencies loaded")
display(df_exchange_rates)

---
## Cell 3 — Status Normalization Mapping

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  STATUS NORMALIZATION
#  Standardizes status values across all channels:
#    → SUCCESS | FAILED | PENDING | REVERSED
# ══════════════════════════════════════════════════════════════════════════

# Card transactions use response_code:
#   "00" = Approved (SUCCESS), "05" = Declined (FAILED),
#   "51" = Insufficient funds (FAILED), others mapped accordingly
card_status_expr = (
    F.when(F.col("response_code") == "00", "SUCCESS")
     .when(F.col("response_code").isin("05", "14", "51", "54", "57", "62"), "FAILED")
     .when(F.col("response_code").isin("01", "02"), "PENDING")
     .when(F.col("response_code") == "34", "REVERSED")
     .otherwise("FAILED")  # Unknown codes default to FAILED
)

# UPI transactions already have status: SUCCESS / FAILED / PENDING
# Just normalize casing and add REVERSED support
upi_status_expr = (
    F.when(F.upper(F.col("status")) == "SUCCESS", "SUCCESS")
     .when(F.upper(F.col("status")) == "FAILED", "FAILED")
     .when(F.upper(F.col("status")) == "PENDING", "PENDING")
     .when(F.upper(F.col("status")).isin("REVERSED", "REFUND", "REVERSAL"), "REVERSED")
     .otherwise("PENDING")
)

# NEFT/RTGS — settled transactions are SUCCESS by default
# (these are cleared through RBI SFMS, so they're already settled)
neft_status_literal = F.lit("SUCCESS")

print("Status normalization expressions defined.")
print("  Card   : response_code → SUCCESS/FAILED/PENDING/REVERSED")
print("  UPI    : status → normalized casing")
print("  NEFT   : all settled → SUCCESS")

---
## Cell 4 — Transform: Card Transactions

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  CHANNEL 1: CARD TRANSACTIONS
#  Source: silver.card_transactions_masked
#  Special handling: currency conversion for international txns
# ══════════════════════════════════════════════════════════════════════════

print(f"Reading: {CARD_TXN_TABLE}")
df_card = spark.read.table(CARD_TXN_TABLE)
print(f"Card transaction count: {df_card.count():,}")

# Join with exchange rates for currency conversion
df_card_unified = (
    df_card
    .join(
        df_exchange_rates,
        df_card.txn_currency == df_exchange_rates.currency_code,
        how="left"
    )
    .select(
        # ── Common schema columns ──
        F.col("txn_id").alias("txn_id"),
        F.col("customer_id").alias("customer_id"),
        F.lit("CARD").alias("txn_channel"),
        # Currency conversion: amount × exchange rate → INR
        F.round(
            F.col("txn_amount") * F.coalesce(F.col("rate_to_inr"), F.lit(1.0)),
            2
        ).alias("txn_amount_inr"),
        F.col("txn_ts").alias("txn_ts"),
        card_status_expr.alias("status"),
        # Masked counterparty = merchant name
        F.col("merchant_name").alias("masked_counterparty"),
        F.col("merchant_id").alias("merchant_id"),
        F.col("is_international").alias("is_international"),
        # ── Extra context columns ──
        F.col("txn_currency").alias("original_currency"),
        F.col("txn_amount").alias("original_amount"),
        F.col("card_number").alias("masked_instrument"),  # Already masked from Task 2.1
        F.col("country_code").alias("country_code"),       # Actual transaction country
    )
)

print(f"Card transactions standardized: {df_card_unified.count():,}")
display(df_card_unified.limit(3))

---
## Cell 5 — Transform: UPI Transactions

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  CHANNEL 2: UPI TRANSACTIONS
#  Source: silver.upi_transactions_masked
#  UPI is always INR, always domestic
# ══════════════════════════════════════════════════════════════════════════

print(f"Reading: {UPI_TXN_TABLE}")
df_upi = spark.read.table(UPI_TXN_TABLE)
print(f"UPI transaction count: {df_upi.count():,}")

df_upi_unified = (
    df_upi
    .select(
        # ── Common schema columns ──
        F.col("upi_txn_id").alias("txn_id"),
        # UPI lacks customer_id; derive from payer_vpa (hash for consistency)
        F.sha2(F.col("payer_vpa"), 256).substr(1, 16).alias("customer_id"),
        F.lit("UPI").alias("txn_channel"),
        # UPI is always INR — no conversion needed
        F.round(F.col("amount"), 2).alias("txn_amount_inr"),
        F.col("txn_ts").alias("txn_ts"),
        upi_status_expr.alias("status"),
        # Masked counterparty = payee VPA
        F.col("payee_vpa").alias("masked_counterparty"),
        F.lit(None).cast(StringType()).alias("merchant_id"),  # UPI has no merchant_id
        F.lit(False).alias("is_international"),               # UPI is domestic only
        # ── Extra context columns ──
        F.lit("INR").alias("original_currency"),
        F.col("amount").alias("original_amount"),
        F.col("payer_vpa").alias("masked_instrument"),
        F.lit("IN").alias("country_code"),                  # UPI is domestic only
    )
)

print(f"UPI transactions standardized: {df_upi_unified.count():,}")
display(df_upi_unified.limit(3))

---
## Cell 6 — Transform: NEFT/RTGS Transactions

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  CHANNEL 3: NEFT / RTGS TRANSACTIONS
#  Source: silver.neft_rtgs_transactions_clean
#  Settled interbank transfers — always INR, always domestic
# ══════════════════════════════════════════════════════════════════════════

print(f"Reading: {NEFT_RTGS_TABLE}")
df_neft = spark.read.table(NEFT_RTGS_TABLE)
print(f"NEFT/RTGS transaction count: {df_neft.count():,}")

df_neft_unified = (
    df_neft
    .select(
        # ── Common schema columns ──
        F.col("instr_id").alias("txn_id"),
        # NEFT/RTGS lacks customer_id; derive from debtor_name (pseudonymized)
        F.sha2(F.col("debtor_name"), 256).substr(1, 16).alias("customer_id"),
        F.lit("NEFT_RTGS").alias("txn_channel"),
        # NEFT/RTGS in INR — no conversion needed
        F.round(F.col("amount"), 2).alias("txn_amount_inr"),
        # NEFT/RTGS has _load_ts as proxy for txn timestamp
        F.col("_load_ts").alias("txn_ts"),
        neft_status_literal.alias("status"),
        # Masked counterparty = creditor_name (already pseudonymized in Task 2.1)
        F.col("creditor_name").alias("masked_counterparty"),
        F.lit(None).cast(StringType()).alias("merchant_id"),
        F.lit(False).alias("is_international"),
        # ── Extra context columns ──
        F.coalesce(F.col("currency"), F.lit("INR")).alias("original_currency"),
        F.col("amount").alias("original_amount"),
        F.col("debtor_name").alias("masked_instrument"),  # Pseudonymized debtor
        F.lit("IN").alias("country_code"),                  # NEFT/RTGS is domestic only
    )
)

print(f"NEFT/RTGS transactions standardized: {df_neft_unified.count():,}")
display(df_neft_unified.limit(3))

---
## Cell 7 — Union All Channels

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  UNION ALL THREE CHANNELS INTO unified_transactions
# ══════════════════════════════════════════════════════════════════════════

df_unified = (
    df_card_unified
    .unionByName(df_upi_unified)
    .unionByName(df_neft_unified)
)

print(f"Total unified records (before dedup): {df_unified.count():,}")
print(f"  Card     : {df_card_unified.count():,}")
print(f"  UPI      : {df_upi_unified.count():,}")
print(f"  NEFT/RTGS: {df_neft_unified.count():,}")

---
## Cell 8 — Cross-Source Deduplication

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  CROSS-SOURCE DEDUPLICATION
#  Detects transactions appearing in multiple channels (e.g., card swipe
#  + UPI for the same payment). Uses fuzzy matching on:
#    - Same customer_id + similar amount + close timestamp
#  Flags duplicates with is_potential_duplicate = True
#  Keeps ALL records (no deletion) — downstream Gold layer decides.
# ══════════════════════════════════════════════════════════════════════════

from pyspark.sql.window import Window

# Step 1: Create a dedup signature
# Transactions within same customer, same amount (±1 INR), within 5-minute window

df_with_dedup = (
    df_unified
    # Create a dedup key: customer_id + rounded amount + 5-min time bucket
    .withColumn(
        "_dedup_amount_bucket",
        F.round(F.col("txn_amount_inr"), 0)  # Round to nearest rupee
    )
    .withColumn(
        "_dedup_time_bucket",
        # 5-minute time windows (300 seconds)
        F.floor(F.unix_timestamp("txn_ts") / 300)
    )
)

# Step 2: Count occurrences within each dedup bucket
dedup_window = Window.partitionBy(
    "customer_id", "_dedup_amount_bucket", "_dedup_time_bucket"
)

df_dedup_flagged = (
    df_with_dedup
    .withColumn("_bucket_count", F.count("*").over(dedup_window))
    .withColumn(
        "is_potential_duplicate",
        # Flag as potential duplicate if same customer+amount+time appears
        # across DIFFERENT channels
        F.when(
            (F.col("_bucket_count") > 1) &
            (F.count(F.col("txn_channel")).over(dedup_window) >
             F.approx_count_distinct(F.col("txn_channel")).over(dedup_window)),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    # Clean up temp columns
    .drop("_dedup_amount_bucket", "_dedup_time_bucket", "_bucket_count")
)

dup_count = df_dedup_flagged.filter(F.col("is_potential_duplicate") == True).count()
print(f"Potential cross-source duplicates flagged: {dup_count:,}")
print(f"Total records after dedup flagging: {df_dedup_flagged.count():,}")
print("(All records kept — duplicates flagged, not removed)")

---
## Cell 9 — Write Unified Transactions to Silver

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  WRITE unified_transactions TO SILVER DELTA TABLE
# ══════════════════════════════════════════════════════════════════════════

# Add final metadata columns
df_final = (
    df_dedup_flagged
    .withColumn("_silver_load_ts", F.current_timestamp())
    .withColumn("_unification_version", F.lit("v1.0-task2.2"))
)

# Write to Delta
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(UNIFIED_TABLE)

print(f"\n Silver table written: {UNIFIED_TABLE}")
print(f"Total unified records: {spark.read.table(UNIFIED_TABLE).count():,}")

---
## Cell 10 — Verification & Summary

In [0]:
# ══════════════════════════════════════════════════════════════════════════
#  TASK 2.2 — VERIFICATION & SUMMARY
# ══════════════════════════════════════════════════════════════════════════

df_result = spark.read.table(UNIFIED_TABLE)

print("=" * 70)
print("  TASK 2.2: UNIFIED TRANSACTIONS — COMPLETE")
print("=" * 70)

# ── Channel distribution ──
print("\n Channel Distribution:")
display(
    df_result
    .groupBy("txn_channel")
    .agg(
        F.count("*").alias("record_count"),
        F.round(F.sum("txn_amount_inr"), 2).alias("total_amount_inr"),
        F.round(F.avg("txn_amount_inr"), 2).alias("avg_amount_inr"),
    )
    .orderBy("txn_channel")
)

# ── Status distribution ──
print("\n Status Distribution:")
display(
    df_result
    .groupBy("txn_channel", "status")
    .agg(F.count("*").alias("count"))
    .orderBy("txn_channel", "status")
)

# ── International transactions ──
intl_count = df_result.filter(F.col("is_international") == True).count()
print(f"\n International transactions: {intl_count:,}")

# ── Dedup summary ──
dup_flagged = df_result.filter(F.col("is_potential_duplicate") == True).count()
print(f" Potential duplicates flagged: {dup_flagged:,}")

# ── Schema validation ──
expected_cols = [
    "txn_id", "customer_id", "txn_channel", "txn_amount_inr",
    "txn_ts", "status", "masked_counterparty", "merchant_id",
    "is_international"
]
actual_cols = df_result.columns
missing = [c for c in expected_cols if c not in actual_cols]

if not missing:
    print("\n Schema validation: ALL required columns present")
else:
    print(f"\n Schema validation: MISSING columns: {missing}")

print(f"\n Total columns: {len(actual_cols)}")
print(f" Total records: {df_result.count():,}")
print(f" Timestamp: {datetime.now().isoformat()}")
print("=" * 70)

---

### Schema: `unified_transactions`

| Column | Type | Source Mapping |
|--------|------|----------------|
| `txn_id` | STRING | Card: `txn_id`, UPI: `upi_txn_id`, NEFT: `instr_id` |
| `customer_id` | STRING | Card: direct, UPI/NEFT: SHA-256 derived |
| `txn_channel` | STRING | `CARD` / `UPI` / `NEFT_RTGS` |
| `txn_amount_inr` | DOUBLE | FX-converted for international card txns |
| `txn_ts` | TIMESTAMP | Card/UPI: `txn_ts`, NEFT: `_load_ts` |
| `status` | STRING | Normalized: `SUCCESS`/`FAILED`/`PENDING`/`REVERSED` |
| `masked_counterparty` | STRING | Card: merchant, UPI: payee VPA, NEFT: creditor |
| `merchant_id` | STRING | Card: direct, UPI/NEFT: NULL |
| `is_international` | BOOLEAN | Card: from data, UPI/NEFT: always `false` |
| `original_currency` | STRING | Pre-conversion currency code |
| `original_amount` | DOUBLE | Pre-conversion amount |
| `masked_instrument` | STRING | Card: masked PAN, UPI: payer VPA, NEFT: debtor |
| `country_code` | STRING | Card: from Bronze data, UPI/NEFT: `IN` (domestic) |
| `is_potential_duplicate` | BOOLEAN | Cross-source dedup flag |

> **Next:** Task 2.3 — Fraud Feature Engineering